# Incident Investigation

Level 5: Senior FDE / Staff-Level Scenarios.

**Problems in this notebook**

- Problem 41: Group Customer Incidents by Root Cause
- Problem 50: Incident Timeline Reconstruction

**How to use this notebook**

1. Read the problem and say your plan out loud before you write any code.
2. Use the AI prompt given (or your own narrow one) to fill the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference only after you have tried.

The ideas behind these problems are in `03_incident_investigation_THEORY.md`.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 41: Group Customer Incidents by Root Cause

**The situation**

Several customers report failures in an AI assistant this week. Support tickets name the root cause in free text, typed by different people. The platform team wants incidents grouped by cause so they can fix the biggest problem first.

**Your task**

Write `group_incidents_by_root_cause(incidents)` that returns a dict from a cleaned root cause to the list of incident ids, in input order.

**Example**

`[{"id": "i1", "root_cause": "Vector DB Timeout"}, {"id": "i2", "root_cause": " vector db  timeout"}]` -> `{"vector db timeout": ["i1", "i2"]}`

**Ask before coding**

- Is the root cause typed by people or set by a system?
- Should similar wording, like "vector DB timeout" and "VDB timed out", be merged?
- Where do incidents with no root cause go?

**Plan to say aloud**

1. Skip any incident with no id, since we cannot point to it.
2. Clean the cause: lowercase, trim, single spaces.
3. A missing or blank cause becomes "unknown".
4. Append the id to that cause's list.

### Prompt for your AI assistant

```text
Write a Python function group_incidents_by_root_cause(incidents: list[dict]) -> dict[str, list[str]].
Rules: skip incidents with no "id". Clean root_cause by lowercasing, trimming and collapsing
inner whitespace to one space. A missing, None, non-string or blank root_cause goes under "unknown".
Keep ids in input order. Standard library only. Show only the function.
```

**Check the AI's code for**

- Calls `.strip()` on a `None` root cause and crashes.
- Uses `incident["id"]`, so one bad record breaks the whole report.
- Does not collapse inner spaces, so `"vector db  timeout"` becomes its own group.
- Puts blank causes under `""` instead of `"unknown"`.

In [ ]:
def group_incidents_by_root_cause(incidents: list[dict]) -> dict[str, list[str]]:
    """Group incident ids by cleaned root cause.

    Rules:
    - skip incidents with no id
    - clean root_cause: lowercase, trim, collapse inner whitespace
    - missing, None, non-string or blank root_cause -> "unknown"
    - keep ids in input order
    """
    raise NotImplementedError

In [ ]:
cases = [
    (([{"id": "i1", "root_cause": "Vector DB Timeout"}],), {"vector db timeout": ["i1"]}),
    (([{"id": "i1", "root_cause": "Vector DB Timeout"},
       {"id": "i2", "root_cause": "  vector db   timeout "}],), {"vector db timeout": ["i1", "i2"]}),
    (([{"id": "i1"}, {"id": "i2", "root_cause": "  "}, {"id": "i3", "root_cause": None}],),
     {"unknown": ["i1", "i2", "i3"]}),
    (([{"root_cause": "Timeout"}, {"id": "i2", "root_cause": "Timeout"}],), {"timeout": ["i2"]}),  # no id: skip
    (([],), {}),
]
run_checks(group_incidents_by_root_cause, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def group_incidents_by_root_cause_reference(incidents: list[dict]) -> dict[str, list[str]]:
    groups: dict[str, list[str]] = {}
    for incident in incidents:
        incident_id = incident.get("id")
        if not incident_id:
            continue
        cause = incident.get("root_cause")
        cause = " ".join(cause.lower().split()) if isinstance(cause, str) else ""
        groups.setdefault(cause or "unknown", []).append(incident_id)
    return groups


assert run_checks(group_incidents_by_root_cause_reference, cases)

**Explain it in 30 seconds**

> "People type root causes in different ways, so I clean the text before grouping. Incidents with no cause go under "unknown" so they stay visible, and records with no id are skipped. The result is a map from cause to incident ids. Next I would sort the groups by customer impact and severity, not just by count."

**Edge cases to mention**

- Missing, blank or `None` root cause.
- Same cause in different words, like "timeout" and "timed out".
- One incident with two causes.
- One loud customer filing many incidents for the same cause.

**If they ask "how would this work in production?"**

- Use a fixed list of cause categories, or embeddings, to merge different wordings of the same cause.
- Rank groups by severity and number of customers affected, and link each to its postmortem.
- Keep tenant ids on each incident so the team can see how many customers each cause hit.

## Problem 50: Incident Timeline Reconstruction

**The situation**

A failed request touched three services: the retriever, the model API and a CRM connector. Each service writes its own log events. Engineers need one clean, ordered timeline per request to see what went wrong first.

**Your task**

Write `reconstruct_incident_timeline(events)` that groups events by `request_id` and returns each request's event names sorted by timestamp `ts`. Skip broken events and drop exact duplicates.

**Example**

`[{"request_id": "r1", "ts": 2, "event": "model_timeout"}, {"request_id": "r1", "ts": 1, "event": "retrieval_started"}]` -> `{"r1": ["retrieval_started", "model_timeout"]}`

**Ask before coding**

- Are timestamps from different services on the same clock?
- Should we group by request, by tenant, or by incident?
- Should repeated identical events be removed?

**Plan to say aloud**

1. Skip events missing request_id, ts or event name.
2. Drop events with the same request_id, ts and name as one we already kept.
3. Group the rest by request_id.
4. Sort each group by ts and keep only the names.

### Prompt for your AI assistant

```text
Write a Python function reconstruct_incident_timeline(events: list[dict]) -> dict[str, list[str]].
Rules: skip events where request_id or ts is None/missing or event is missing/empty.
A ts of 0 is valid. Drop exact duplicates (same request_id, ts and event).
Group by request_id, sort each group by ts (stable, so ties keep input order), and return
the event names. Standard library only. Show only the function.
```

**Check the AI's code for**

- Uses `if not event.get("ts")`, which drops a valid timestamp of `0`.
- Sorts all events globally but never groups by request.
- Removes duplicates by event name only, which deletes real repeated steps at different times.
- Crashes with `KeyError` on an event with no `ts`.

In [ ]:
def reconstruct_incident_timeline(events: list[dict]) -> dict[str, list[str]]:
    """Per-request timeline of event names, sorted by ts.

    Rules:
    - skip events with missing/None request_id or ts, or missing/empty event (ts=0 is valid)
    - drop exact duplicates: same request_id, ts and event
    - group by request_id, sort by ts (ties keep input order), return event names
    """
    raise NotImplementedError

In [ ]:
cases = [
    (([{"request_id": "r1", "ts": 2, "event": "model_timeout"},
       {"request_id": "r1", "ts": 1, "event": "retrieval_started"}],),
     {"r1": ["retrieval_started", "model_timeout"]}),
    (([{"request_id": "r1", "ts": 3, "event": "crm_error"},
       {"request_id": "r2", "ts": 1, "event": "retrieval_started"},
       {"request_id": "r1", "ts": 1, "event": "retrieval_started"}],),
     {"r1": ["retrieval_started", "crm_error"], "r2": ["retrieval_started"]}),
    (([{"request_id": "r1", "event": "no_timestamp"},
       {"ts": 1, "event": "no_request"},
       {"request_id": "r1", "ts": 0, "event": "received"}],), {"r1": ["received"]}),  # ts=0 is valid
    (([{"request_id": "r1", "ts": 1, "event": "retry"},
       {"request_id": "r1", "ts": 1, "event": "retry"},
       {"request_id": "r1", "ts": 2, "event": "retry"}],), {"r1": ["retry", "retry"]}),  # exact duplicate dropped
    (([],), {}),
]
run_checks(reconstruct_incident_timeline, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def reconstruct_incident_timeline_reference(events: list[dict]) -> dict[str, list[str]]:
    seen = set()
    grouped: dict[str, list[dict]] = {}
    for e in events:
        if e.get("request_id") is None or e.get("ts") is None or not e.get("event"):
            continue
        fingerprint = (e["request_id"], e["ts"], e["event"])
        if fingerprint in seen:
            continue
        seen.add(fingerprint)
        grouped.setdefault(e["request_id"], []).append(e)
    return {
        rid: [e["event"] for e in sorted(items, key=lambda e: e["ts"])]
        for rid, items in grouped.items()
    }


assert run_checks(reconstruct_incident_timeline_reference, cases)

**Explain it in 30 seconds**

> "A timeline is only as good as the request ids and timestamps the services emit. I skip events I cannot place, drop exact repeats, group by request id, and sort each group by time. That gives engineers one clear story per request. If two services have clocks that disagree, the order can still be wrong, so I would say that out loud."

**Edge cases to mention**

- A timestamp of 0, which is valid but falsy in Python.
- The same event logged twice by a retrying service.
- Clock skew: two servers whose clocks differ by a few seconds.
- Events from two tenants sharing a request id by accident.

**If they ask "how would this work in production?"**

- Use distributed tracing, where each request carries a trace id and each step is a span with its own id and parent.
- Put tenant id and service name on every event, so the timeline can be filtered safely.
- Feed the timelines into a dashboard so on-call engineers can see the slow or failing step.

## Wrap-up

- Logs and tickets are messy. Clean and validate before you group or sort.
- Skip records you cannot place, but keep unknowns visible instead of hiding them.
- Good incident tools depend on good ids: incident ids, request ids, tenant ids.